![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/18.0.Chunk_Sentence_Splitter.ipynb)

If you are using the `spark-nlp-jsl` library, please use this  [18.Chunk_Sentence_Splitter](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/18.Chunk_Sentence_Splitter.ipynb) notebook.

# Chunk Sentence Splitter
We are releasing `ChunkSentenceSplitter`  annotator that splits documents or sentences by chunks provided. Splitted parts can be named with the splitting chunks. <br/>
By using this annotator, you can do some tasks like splitting clinical documents according into sections in accordance with CDA (Clinical Document Architecture).

## Healthcare NLP for Data Scientists Course

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each components.

## Colab Setup

In [ ]:
# Install the johnsnowlabs library to access Spark-OCR and Spark-NLP for Healthcare, Finance, and Legal.
! pip install -q johnsnowlabs

In [ ]:
from google.colab import files
print('Please Upload your John Snow Labs License using the button below')
license_keys = files.upload()

In [ ]:
from johnsnowlabs import nlp, medical

# After uploading your license run this to install all licensed Python Wheels and pre-download Jars the Spark Session JVM
nlp.install()

In [ ]:
from johnsnowlabs import nlp, medical

# Automatically load license data and start a session with all jars user has access to
spark = nlp.start()

In [5]:
spark

## How It Works


In [6]:
#giving input chunks to the ChunkSentenceSplitter model by using regex
regex = """Reporting Template,title1
SPECIMEN,title2
RESULTS,title3"""

with open("title_regex.txt", 'w') as f:
  f.write(regex)

In [7]:
from pyspark.sql.types import StringType

In [8]:
import pandas as pd

documentAssembler = nlp.DocumentAssembler()\
     .setInputCol("text")\
     .setOutputCol("document")

regexMatcher = nlp.RegexMatcher()\
     .setExternalRules("/content/title_regex.txt", ",")\
     .setInputCols("document")\
     .setOutputCol("chunks")

pipeline =  nlp.Pipeline().setStages([
                                  documentAssembler,
                                  regexMatcher])

text_list = ["""
This is the header that have not title

Reporting Template

Writers write descriptive paragraphs because their purpose is to describe something. Their point is that something
is beautiful or disgusting or strangely intriguing.
Writers write persuasive and argument paragraphs because their purpose is to persuade or convince someone. T

SPECIMEN
+Adequacy of Sample for Testing
___ Adequate
+Estimated % Tumor Cellularity
___ Suboptimal (explain): _________________

RESULTS
+Mutational Analysis
___ Mutation detected
___ Mutation no identified
___ EGFR
"""]

# data_chunk = spark.createDataFrame([["text"]]).toDF("text")

# pipeline_model = pipeline.fit(data_chunk)

# chunk_df = pipeline_model.transform(spark.createDataFrame(pd.DataFrame({'text': text_list}).values.tolist(), schema=['text']))

data_chunk = spark.createDataFrame([["text"]], schema=StringType()).toDF("text")

pipeline_model = pipeline.fit(data_chunk)

chunk_df = pipeline_model.transform(spark.createDataFrame(text_list, StringType()).toDF("text"))


In [9]:
chunk_df.show()

+--------------------+--------------------+--------------------+
|                text|            document|              chunks|
+--------------------+--------------------+--------------------+
|\nThis is the hea...|[{document, 0, 55...|[{chunk, 41, 58, ...|
+--------------------+--------------------+--------------------+



In [10]:
chunk_df.selectExpr('explode(chunks)').show(truncate=False)

+------------------------------------------------------------------------------------------+
|col                                                                                       |
+------------------------------------------------------------------------------------------+
|{chunk, 41, 58, Reporting Template, {identifier -> title1, sentence -> 0, chunk -> 0}, []}|
|{chunk, 338, 345, SPECIMEN, {identifier -> title2, sentence -> 0, chunk -> 1}, []}        |
|{chunk, 468, 474, RESULTS, {identifier -> title3, sentence -> 0, chunk -> 2}, []}         |
+------------------------------------------------------------------------------------------+



Applying `ChunkSentenceSplitter()`

In [11]:
chunkSentenceSplitter = medical.ChunkSentenceSplitter()\
      .setInputCols("chunks","document")\
      .setOutputCol("paragraphs")

paragraphs = chunkSentenceSplitter.transform(chunk_df)

In [12]:
paragraphs.selectExpr("explode(paragraphs) as result").selectExpr("result.result","result.metadata.entity").toPandas()

,result,entity
0,\nThis is the header that have not title\n\n,introduction
1,Reporting Template\n\nWriters write descriptiv...,title1


### Ner Pipeline with Sentence Splitting

In [13]:
#input data

input_list = ["""Sample Name: Mesothelioma - Pleural Biopsy
Description: Right pleural effusion and suspected malignant mesothelioma. (Medical Transcription Sample Report)
PREOPERATIVE DIAGNOSIS:  Right pleural effusion and suspected malignant mesothelioma.
POSTOPERATIVE DIAGNOSIS: Right pleural effusion, suspected malignant mesothelioma.
ANESTHESIA: General double-lumen endotracheal.
DESCRIPTION OF FINDINGS:  Right pleural effusion, firm nodules, diffuse scattered throughout the right pleura and diaphragmatic surface.
SPECIMEN:  Pleural biopsies for pathology and microbiology.
INDICATIONS:  Briefly, this is a 66-year-old gentleman who has been transferred from an outside hospital after a pleural effusion had been drained and biopsies taken from the right chest that were thought to be consistent with mesothelioma. Upon transfer, he had a right pleural effusion demonstrated on x-ray as well as some shortness of breath and dyspnea on exertion. The risks, benefits, and alternatives to right VATS pleurodesis and pleural biopsy were discussed with the patient and his family and they wished to proceed.
Dr. X was present for the entire procedure which was right VATS pleurodesis and pleural biopsies.The counts were correct x2 at the end of the case."""]

In [14]:
files = [f"{i}.txt" for i in (range(1, len(input_list)+1))]

df = spark.createDataFrame(pd.DataFrame({'text': input_list, 'file': files}).values.tolist(), schema=['text', 'file'])

df.show()

+--------------------+-----+
|                text| file|
+--------------------+-----+
|Sample Name: Meso...|1.txt|
+--------------------+-----+



Now, creating NER pipeline for extracting chunks

In [15]:
documentAssembler = nlp.DocumentAssembler()\
      .setInputCol("text")\
      .setOutputCol("document")

sentenceDetector = nlp.SentenceDetectorDLModel.pretrained("sentence_detector_dl_healthcare","en","clinical/models")\
        .setInputCols(["document"])\
        .setOutputCol("sentence")

tokenizer = nlp.Tokenizer()\
      .setInputCols(["sentence"])\
      .setOutputCol("token")\

word_embeddings = nlp.WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
      .setInputCols(["sentence", "token"])\
      .setOutputCol("embeddings")

clinical_ner = medical.NerModel.pretrained("ner_jsl_slim", "en", "clinical/models") \
      .setInputCols(["sentence", "token", "embeddings"]) \
      .setOutputCol("ner")

ner_converter = medical.NerConverterInternal() \
      .setInputCols(["sentence", "token", "ner"]) \
      .setOutputCol("ner_chunk")\
      .setWhiteList(["Header"])

pipeline_sentence = nlp.Pipeline(
    stages = [
        documentAssembler,
        sentenceDetector,
        tokenizer,
        word_embeddings,
        clinical_ner,
        ner_converter
    ])

empty_df = spark.createDataFrame([[""]]).toDF('text')
pipeline_model_sentence = pipeline_sentence.fit(empty_df)

sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
ner_jsl_slim download started this may take some time.
Approximate size to download 14.4 MB
[OK!]


In [16]:
result = pipeline_model_sentence.transform(df)
result.selectExpr('explode(ner_chunk)').show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+
|col                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|{chunk, 43, 54, Description:, {entity -> Header, confidence -> 0.8571, ner_source -> ner_chunk, chunk -> 0, sentence -> 1}, []}                 |
|{chunk, 155, 177, PREOPERATIVE DIAGNOSIS:, {entity -> Header, confidence -> 0.87280005, ner_source -> ner_chunk, chunk -> 1, sentence -> 3}, []}|
|{chunk, 241, 264, POSTOPERATIVE DIAGNOSIS:, {entity -> Header, confidence -> 0.8618, ner_source -> ner_chunk, chunk -> 2, sentence -> 4}, []}   |
|{chunk, 324, 334, ANESTHESIA:, {entity -> Header, confidence -> 0.68285, ner_source -> ner_chunk, chunk -> 3, sentenc

In [17]:
result.columns

['text',
 'file',
 'document',
 'sentence',
 'token',
 'embeddings',
 'ner',
 'ner_chunk']

In [18]:
#applying ChunkSentenceSplitter
chunkSentenceSplitter = medical.ChunkSentenceSplitter()\
    .setInputCols("document","ner_chunk")\
    .setOutputCol("paragraphs")\
    .setGroupBySentences(False)

paragraphs = chunkSentenceSplitter.transform(result)

In [19]:
paragraphs.show()

+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text| file|            document|            sentence|               token|          embeddings|                 ner|           ner_chunk|          paragraphs|
+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Sample Name: Meso...|1.txt|[{document, 0, 12...|[{document, 0, 41...|[{token, 0, 5, Sa...|[{word_embeddings...|[{named_entity, 0...|[{chunk, 43, 54, ...|[{document, 0, 43...|
+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+



In [20]:
paragraphs.select("paragraphs.result").show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                              result|
+----------------------------------------------------------------------------------------------------+
|[Sample Name: Mesothelioma - Pleural Biopsy\n, Description: Right pleural effusion and suspected ...|
+----------------------------------------------------------------------------------------------------+



In [21]:
pd.set_option('display.max_colwidth', None)
result_df = paragraphs.selectExpr("explode(paragraphs) as result").selectExpr("result.result","result.metadata.entity").toPandas()
result_df.head()

,result,entity
0,Sample Name: Mesothelioma - Pleural Biopsy\n,introduction
1,Description: Right pleural effusion and suspected malignant mesothelioma. (Medical Transcription Sample Report)\n,Header
2,PREOPERATIVE DIAGNOSIS: Right pleural effusion and suspected malignant mesothelioma.\n,Header
3,"POSTOPERATIVE DIAGNOSIS: Right pleural effusion, suspected malignant mesothelioma.\n",Header
4,ANESTHESIA: General double-lumen endotracheal.\n,Header


### Ner Pipeline without Sentence Splitter

In [22]:
documentAssembler = nlp.DocumentAssembler()\
      .setInputCol("text")\
      .setOutputCol("document")

#sentenceDetector = SentenceDetectorDLModel.pretrained("sentence_detector_dl_healthcare","en","clinical/models")\
#        .setInputCols(["document"])\
#        .setOutputCol("sentence")

tokenizer= nlp.Tokenizer()\
      .setInputCols(["document"])\
      .setOutputCol("token")

tokenClassifier = medical.BertForTokenClassification.pretrained("bert_token_classifier_ner_jsl_slim", "en", "clinical/models")\
    .setInputCols("token", "document")\
    .setOutputCol("ner")\
    .setCaseSensitive(True)

ner_converter = medical.NerConverterInternal() \
      .setInputCols(["document", "token", "ner"]) \
      .setOutputCol("ner_chunk")\
      .setWhiteList(["Header"])

pipeline = nlp.Pipeline(
    stages = [
        documentAssembler,
        tokenizer,
        tokenClassifier,
        ner_converter
    ])

empty_df = spark.createDataFrame([[""]]).toDF('text')
pipeline_model = pipeline.fit(empty_df)

bert_token_classifier_ner_jsl_slim download started this may take some time.
Approximate size to download 385.7 MB
[OK!]


In [23]:
result = pipeline_model.transform(df)
result.selectExpr('explode(ner_chunk)').show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+
|col                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|{chunk, 155, 177, PREOPERATIVE DIAGNOSIS:, {entity -> Header, confidence -> 0.9747639, ner_source -> ner_chunk, chunk -> 0, sentence -> 0}, []} |
|{chunk, 241, 264, POSTOPERATIVE DIAGNOSIS:, {entity -> Header, confidence -> 0.9736075, ner_source -> ner_chunk, chunk -> 1, sentence -> 0}, []}|
|{chunk, 324, 333, ANESTHESIA, {entity -> Header, confidence -> 0.62232125, ner_source -> ner_chunk, chunk -> 2, sentence -> 0}, []}             |
|{chunk, 371, 393, DESCRIPTION OF FINDINGS, {entity -> Header, confidence -> 0.99747926, ner_source -> ner_chunk, chun

In [24]:
result.columns #no sentence column

['text', 'file', 'document', 'token', 'ner', 'ner_chunk']

In [25]:
#applying ChunkSentenceSplitter
chunkSentenceSplitter = medical.ChunkSentenceSplitter()\
    .setInputCols("ner_chunk","document")\
    .setOutputCol("paragraphs")\

paragraphs = chunkSentenceSplitter.transform(result)

In [26]:
paragraphs.show()

+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text| file|            document|               token|                 ner|           ner_chunk|          paragraphs|
+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|Sample Name: Meso...|1.txt|[{document, 0, 12...|[{token, 0, 5, Sa...|[{named_entity, 0...|[{chunk, 155, 177...|[{document, 0, 15...|
+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+



In [27]:
paragraphs.select("paragraphs.result").show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                              result|
+----------------------------------------------------------------------------------------------------+
|[Sample Name: Mesothelioma - Pleural Biopsy\nDescription: Right pleural effusion and suspected ma...|
+----------------------------------------------------------------------------------------------------+



In [28]:
result_df = paragraphs.selectExpr("explode(paragraphs) as result").selectExpr("result.result","result.metadata.entity", "result.metadata.splitter_chunk").toPandas()
result_df.head()

,result,entity,splitter_chunk
0,Sample Name: Mesothelioma - Pleural Biopsy\nDescription: Right pleural effusion and suspected malignant mesothelioma. (Medical Transcription Sample Report)\n,introduction,UNK
1,"PREOPERATIVE DIAGNOSIS: Right pleural effusion and suspected malignant mesothelioma.\nPOSTOPERATIVE DIAGNOSIS: Right pleural effusion, suspected malignant mesothelioma.\nANESTHESIA: General double-lumen endotracheal.\nDESCRIPTION OF FINDINGS: Right pleural effusion, firm nodules, diffuse scattered throughout the right pleura and diaphragmatic surface.\nSPECIMEN: Pleural biopsies for pathology and microbiology.\nINDICATIONS: Briefly, this is a 66-year-old gentleman who has been transferred from an outside hospital after a pleural effusion had been drained and biopsies taken from the right chest that were thought to be consistent with mesothelioma. Upon transfer, he had a right pleural effusion demonstrated on x-ray as well as some shortness of breath and dyspnea on exertion. The risks, benefits, and alternatives to right VATS pleurodesis and pleural biopsy were discussed with the patient and his family and they wished to proceed.\nDr. X was present for the entire procedure which was right VATS pleurodesis and pleural biopsies.The counts were correct x2 at the end of the case.",Header,PREOPERATIVE DIAGNOSIS:


`.setInsertChunk()` parameter to set whether remove chunks from splitted parts or not.

In [29]:
chunkSentenceSplitter = medical.ChunkSentenceSplitter()\
    .setInputCols("ner_chunk","document")\
    .setOutputCol("paragraphs")\
    .setInsertChunk(False)

paragraphs = chunkSentenceSplitter.transform(result)

In [30]:
paragraphs.select("paragraphs.result").show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                              result|
+----------------------------------------------------------------------------------------------------+
|[Sample Name: Mesothelioma - Pleural Biopsy\nDescription: Right pleural effusion and suspected ma...|
+----------------------------------------------------------------------------------------------------+



In [31]:
result_insert = paragraphs.selectExpr("explode(paragraphs) as result").selectExpr("result.result","result.metadata.entity", "result.metadata.splitter_chunk").toPandas()
result_insert.head()

,result,entity,splitter_chunk
0,Sample Name: Mesothelioma - Pleural Biopsy\nDescription: Right pleural effusion and suspected malignant mesothelioma. (Medical Transcription Sample Report)\n,introduction,UNK
1,"Right pleural effusion and suspected malignant mesothelioma.\nPOSTOPERATIVE DIAGNOSIS: Right pleural effusion, suspected malignant mesothelioma.\nANESTHESIA: General double-lumen endotracheal.\nDESCRIPTION OF FINDINGS: Right pleural effusion, firm nodules, diffuse scattered throughout the right pleura and diaphragmatic surface.\nSPECIMEN: Pleural biopsies for pathology and microbiology.\nINDICATIONS: Briefly, this is a 66-year-old gentleman who has been transferred from an outside hospital after a pleural effusion had been drained and biopsies taken from the right chest that were thought to be consistent with mesothelioma. Upon transfer, he had a right pleural effusion demonstrated on x-ray as well as some shortness of breath and dyspnea on exertion. The risks, benefits, and alternatives to right VATS pleurodesis and pleural biopsy were discussed with the patient and his family and they wished to proceed.\nDr. X was present for the entire procedure which was right VATS pleurodesis and pleural biopsies.The counts were correct x2 at the end of the case.",Header,PREOPERATIVE DIAGNOSIS:


Check how `.setInsertChunk(True)` affects the result

In [32]:
chunkSentenceSplitter_2 = medical.ChunkSentenceSplitter()\
    .setInputCols("ner_chunk","document")\
    .setOutputCol("paragraphs")\
    .setInsertChunk(True)\
    .setDefaultEntity("Intro") #to set the name of the introduction entity


paragraphs = chunkSentenceSplitter_2.transform(result)

result = paragraphs.selectExpr("explode(paragraphs) as result").selectExpr("result.result","result.metadata.entity", "result.metadata.splitter_chunk").toPandas()
result.head()

,result,entity,splitter_chunk
0,Sample Name: Mesothelioma - Pleural Biopsy\nDescription: Right pleural effusion and suspected malignant mesothelioma. (Medical Transcription Sample Report)\n,Intro,UNK
1,"PREOPERATIVE DIAGNOSIS: Right pleural effusion and suspected malignant mesothelioma.\nPOSTOPERATIVE DIAGNOSIS: Right pleural effusion, suspected malignant mesothelioma.\nANESTHESIA: General double-lumen endotracheal.\nDESCRIPTION OF FINDINGS: Right pleural effusion, firm nodules, diffuse scattered throughout the right pleura and diaphragmatic surface.\nSPECIMEN: Pleural biopsies for pathology and microbiology.\nINDICATIONS: Briefly, this is a 66-year-old gentleman who has been transferred from an outside hospital after a pleural effusion had been drained and biopsies taken from the right chest that were thought to be consistent with mesothelioma. Upon transfer, he had a right pleural effusion demonstrated on x-ray as well as some shortness of breath and dyspnea on exertion. The risks, benefits, and alternatives to right VATS pleurodesis and pleural biopsy were discussed with the patient and his family and they wished to proceed.\nDr. X was present for the entire procedure which was right VATS pleurodesis and pleural biopsies.The counts were correct x2 at the end of the case.",Header,PREOPERATIVE DIAGNOSIS:
